# 3. Transformer Training

This notebook trains the transformer classifier on proposal-style burst tokens with a BPE tokenizer.

In [ ]:
!python -m pip install -q numpy scikit-learn tokenizers torch

In [ ]:
from pathlib import Path

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    drive = None

if IN_COLAB:
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/final project/dataset')
else:
    PROJECT_DIR = Path.cwd()

print(f'Project directory: {PROJECT_DIR}')

In [ ]:
import json
import random
import tempfile
import warnings
from collections.abc import Callable, Iterable, Iterator, Sequence

import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer
from torch.utils.data import DataLoader, Dataset

In [ ]:
DATASET_NAME = 'tor_100w_2500tr.npz'
N_SITES = 100
N_TRACES = 100
MAX_LEN = 512
BPE_VOCAB_SIZE = 1000
BATCH_SIZE = 32
EPOCHS = 10
LR = 1e-3
D_MODEL = 128
NHEAD = 4
NUM_LAYERS = 4
DROPOUT = 0.1
VAL_SIZE = 0.2
SEED = 42

DATASET_PATH = PROJECT_DIR / DATASET_NAME
ARTIFACT_DIR = PROJECT_DIR / 'artifacts'
ARTIFACT_DIR.mkdir(exist_ok=True)

In [ ]:
def resolve_dataset_path(data_path: str | Path) -> Path:
    path = Path(data_path)
    if path.is_file() and path.suffix.lower() == '.npz':
        return path
    raise ValueError(f'Dataset path does not exist or is unsupported: {path}')


def load_traces(
    data_path: str | Path,
    n_sites: int = 100,
    n_traces: int = 100,
) -> tuple[list[list[int]], list[int]]:
    resolved_path = resolve_dataset_path(data_path)

    with warnings.catch_warnings():
        warnings.filterwarnings(
            'ignore',
            category=np.exceptions.VisibleDeprecationWarning,
        )
        with np.load(resolved_path, allow_pickle=True) as dataset:
            data = dataset['data']
            raw_labels = np.asarray(dataset['labels'])

    unique_sites = np.unique(raw_labels)
    selected_sites = unique_sites[:n_sites]
    traces: list[list[int]] = []
    labels: list[int] = []
    site_to_id = {site: index for index, site in enumerate(selected_sites)}

    for site in selected_sites:
        site_indices = np.where(raw_labels == site)[0][:n_traces]
        for index in site_indices:
            traces.append(data[index].tolist())
            labels.append(site_to_id[site])

    return traces, labels


def burst_direction(value: int) -> str:
    return 'OUT' if value == 1 else 'IN'


def iter_bursts(trace: Sequence[int]) -> Iterator[tuple[str, int]]:
    index = 0
    while index < len(trace):
        current_value = trace[index]
        length = 1
        while index + length < len(trace) and trace[index + length] == current_value:
            length += 1
        yield burst_direction(current_value), length
        index += length


def proposal_length_bucket(length: int) -> str:
    if length <= 16:
        return f'LEN_{length:03d}'
    if length <= 32:
        start = ((length - 17) // 2) * 2 + 17
        end = start + 1
        return f'LEN_{start:03d}_{end:03d}'
    if length <= 64:
        start = ((length - 33) // 4) * 4 + 33
        end = start + 3
        return f'LEN_{start:03d}_{end:03d}'
    if length <= 128:
        start = ((length - 65) // 8) * 8 + 65
        end = start + 7
        return f'LEN_{start:03d}_{end:03d}'
    if length <= 256:
        start = ((length - 129) // 16) * 16 + 129
        end = start + 15
        return f'LEN_{start:03d}_{end:03d}'
    if length <= 512:
        start = ((length - 257) // 32) * 32 + 257
        end = start + 31
        return f'LEN_{start:03d}_{end:03d}'
    if length <= 1024:
        start = ((length - 513) // 64) * 64 + 513
        end = start + 63
        return f'LEN_{start:03d}_{end:03d}'
    return 'LEN_1025_PLUS'


def proposal_burst_tokenize(trace: Sequence[int]) -> list[str]:
    return [
        f'{direction}_{proposal_length_bucket(length)}'
        for direction, length in iter_bursts(trace)
    ]


def traces_to_documents(
    traces: Iterable[Sequence[int]],
    tokenize_fn: Callable[[Sequence[int]], list[str]],
) -> list[str]:
    return [' '.join(tokenize_fn(trace)) for trace in traces]


def train_bpe_tokenizer(documents: list[str], vocab_size: int = 1000) -> Tokenizer:
    tokenizer = Tokenizer(BPE(unk_token='[UNK]'))
    tokenizer.pre_tokenizer = Whitespace()
    trainer = BpeTrainer(vocab_size=vocab_size, special_tokens=['[UNK]', '[CLS]', '[PAD]'])

    with tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', suffix='.txt', delete=False) as handle:
        corpus_path = Path(handle.name)
        for document in documents:
            handle.write(document + '\n')

    try:
        tokenizer.train(files=[str(corpus_path)], trainer=trainer)
    finally:
        corpus_path.unlink(missing_ok=True)

    return tokenizer

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class TrafficDataset(Dataset):
    def __init__(
        self,
        traces: list[list[int]],
        labels: list[int],
        tokenizer: Tokenizer,
        max_len: int = MAX_LEN,
    ) -> None:
        self.input_ids: list[list[int]] = []
        self.attention_masks: list[list[int]] = []
        self.labels = labels

        cls_id = tokenizer.token_to_id('[CLS]')
        pad_id = tokenizer.token_to_id('[PAD]')

        documents = traces_to_documents(traces, tokenize_fn=proposal_burst_tokenize)
        for document in documents:
            token_ids = tokenizer.encode(document).ids[: max_len - 1]
            ids = [cls_id] + token_ids
            attention_mask = [1] * len(ids)

            pad_count = max_len - len(ids)
            ids += [pad_id] * pad_count
            attention_mask += [0] * pad_count

            self.input_ids.append(ids)
            self.attention_masks.append(attention_mask)

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        return (
            torch.tensor(self.input_ids[idx], dtype=torch.long),
            torch.tensor(self.attention_masks[idx], dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.long),
        )


class TrafficTransformer(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        n_classes: int,
        pad_id: int,
        d_model: int = D_MODEL,
        nhead: int = NHEAD,
        num_layers: int = NUM_LAYERS,
        max_len: int = MAX_LEN,
        dropout: float = DROPOUT,
    ) -> None:
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_encoding = nn.Embedding(max_len, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=256,
            dropout=dropout,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Linear(d_model, n_classes)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        positions = torch.arange(input_ids.size(1), device=input_ids.device).unsqueeze(0)
        hidden_states = self.embedding(input_ids) + self.pos_encoding(positions)
        padding_mask = attention_mask == 0
        hidden_states = self.transformer(hidden_states, src_key_padding_mask=padding_mask)
        cls_output = hidden_states[:, 0, :]
        return self.classifier(cls_output)


def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    optimizer: torch.optim.Optimizer | None = None,
) -> tuple[float, float]:
    is_training = optimizer is not None
    model.train() if is_training else model.eval()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    with torch.set_grad_enabled(is_training):
        for input_ids, attention_mask, batch_labels in loader:
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            batch_labels = batch_labels.to(device)

            if optimizer is not None:
                optimizer.zero_grad()

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, batch_labels)

            if optimizer is not None:
                loss.backward()
                optimizer.step()

            batch_size = batch_labels.size(0)
            total_loss += loss.item() * batch_size
            total_correct += (logits.argmax(dim=1) == batch_labels).sum().item()
            total_examples += batch_size

    return total_loss / total_examples, total_correct / total_examples

In [ ]:
set_seed(SEED)
traces, labels = load_traces(DATASET_PATH, n_sites=N_SITES, n_traces=N_TRACES)

train_traces, val_traces, train_labels, val_labels = train_test_split(
    traces,
    labels,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=labels,
)

train_docs = traces_to_documents(train_traces, tokenize_fn=proposal_burst_tokenize)
tokenizer = train_bpe_tokenizer(train_docs, vocab_size=BPE_VOCAB_SIZE)

train_dataset = TrafficDataset(train_traces, train_labels, tokenizer, max_len=MAX_LEN)
val_dataset = TrafficDataset(val_traces, val_labels, tokenizer, max_len=MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
pad_id = tokenizer.token_to_id('[PAD]')
model = TrafficTransformer(
    vocab_size=tokenizer.get_vocab_size(),
    n_classes=len(set(labels)),
    pad_id=pad_id,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

history = []
best_val_acc = 0.0
tokenizer_path = ARTIFACT_DIR / 'traffic_bpe_tokenizer.json'
checkpoint_path = ARTIFACT_DIR / 'traffic_transformer.pt'
summary_path = ARTIFACT_DIR / 'traffic_transformer_summary.json'

print(f'Device: {device}')
print(f'Loaded {len(traces)} traces across {len(set(labels))} sites')
print(f'Train/val split: {len(train_dataset)}/{len(val_dataset)}')
print(f'BPE vocab size: {tokenizer.get_vocab_size()}')

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, device, optimizer=optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, criterion, device)

    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'train_acc': train_acc,
        'val_loss': val_loss,
        'val_acc': val_acc,
    })

    print(
        f'Epoch {epoch}: '
        f'train_loss={train_loss:.3f}, train_acc={train_acc:.3f}, '
        f'val_loss={val_loss:.3f}, val_acc={val_acc:.3f}'
    )

    if val_acc >= best_val_acc:
        best_val_acc = val_acc
        tokenizer.save(str(tokenizer_path))
        torch.save(
            {
                'model_state_dict': model.state_dict(),
                'config': {
                    'vocab_size': tokenizer.get_vocab_size(),
                    'n_classes': len(set(labels)),
                    'pad_id': pad_id,
                    'd_model': D_MODEL,
                    'nhead': NHEAD,
                    'num_layers': NUM_LAYERS,
                    'max_len': MAX_LEN,
                    'dropout': DROPOUT,
                },
            },
            checkpoint_path,
        )

summary = {
    'dataset': DATASET_NAME,
    'n_sites': N_SITES,
    'n_traces': N_TRACES,
    'bpe_vocab_size': tokenizer.get_vocab_size(),
    'best_val_acc': best_val_acc,
    'history': history,
}
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print(f'Best validation accuracy: {best_val_acc:.3f}')
print(f'Saved tokenizer to: {tokenizer_path}')
print(f'Saved model checkpoint to: {checkpoint_path}')
print(f'Saved training summary to: {summary_path}')